# Limpieza y Transformación (ETL) del año 2021
Usando el dataframe ya en limpio del año 2019 decidi contraponer el año 2020 y 2021 para luego al final, cuando cree visualizaciones, poder tener un contexto de pre pandemia, pandemia en si y post pandemia para medir.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

path_clean = "../data/clean/"
path_raw = "../data/raw/"

df_2019_clean = pd.read_csv(
    path_clean + "historico_2019_clean.csv",
    sep=",",
    encoding="utf-8-sig"
)

schema_2019 = df_2019_clean.columns.tolist()

print("Columnas schema 2019:", len(schema_2019))
schema_2019


Columnas schema 2019: 17


['periodo',
 'fecha',
 'desde',
 'hasta',
 'linea',
 'molinete',
 'estacion',
 'pax_pagos',
 'pax_pases_pagos',
 'pax_franq',
 'total',
 'hora_desde',
 'hora_hasta',
 'dia_semana',
 'mes',
 'dia_mes',
 'es_fin_semana']

In [2]:
#Carga datos crudos

df_2021 = pd.read_csv(path_raw + "historico_2021.csv", sep=";")

print(df_2021.shape)
df_2021.head(2)

(8071680, 11)


,periodo,FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,pax_pagos,pax_pases_pagos,pax_franq,pax_TOTAL
0,2021,1/1/2021,08:00:00,08:15:00,LineaA,LineaA_Flores_Este_Turn02,Flores,1,0,0,1
1,2021,1/1/2021,08:00:00,08:15:00,LineaA,LineaA_SanPedrito_Oeste_Turn06,San Pedrito,0,0,2,2


In [3]:
df_2021.columns.tolist()


['periodo',
 'FECHA',
 'DESDE',
 'HASTA',
 'LINEA',
 'MOLINETE',
 'ESTACION',
 'pax_pagos',
 'pax_pases_pagos',
 'pax_franq',
 'pax_TOTAL']

In [4]:
cols_2021 = set(df_2021.columns)
cols_2019 = set(schema_2019)

faltan_en_2021 = sorted(list(cols_2019 - cols_2021))
sobran_en_2021 = sorted(list(cols_2021 - cols_2019))
print("Faltan en 2021:", faltan_en_2021)
print("Sobran en 2021:", sobran_en_2021)

Faltan en 2021: ['desde', 'dia_mes', 'dia_semana', 'es_fin_semana', 'estacion', 'fecha', 'hasta', 'hora_desde', 'hora_hasta', 'linea', 'mes', 'molinete', 'total']
Sobran en 2021: ['DESDE', 'ESTACION', 'FECHA', 'HASTA', 'LINEA', 'MOLINETE', 'pax_TOTAL']


In [5]:
rename_map = {
    "FECHA": "fecha",
    "DESDE": "desde",
    "HASTA": "hasta",
    "LINEA": "linea",
    "MOLINETE": "molinete",
    "ESTACION": "estacion",
    "pax_TOTAL": "total",
}

df_2021 = df_2021.rename(columns=rename_map)
df_2021.columns

Index(['periodo', 'fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total'], dtype='object')

In [6]:
cols_2021 = set(df_2021.columns)
cols_2019 = set(schema_2019)

faltan_en_2021 = sorted(list(cols_2019 - cols_2021))
sobran_en_2021 = sorted(list(cols_2021 - cols_2019))
print("Faltan en 2021:", faltan_en_2021)
print("Sobran en 2021:", sobran_en_2021)

Faltan en 2021: ['dia_mes', 'dia_semana', 'es_fin_semana', 'hora_desde', 'hora_hasta', 'mes']
Sobran en 2021: []


In [7]:
df_2021.head(2)

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,2021,1/1/2021,08:00:00,08:15:00,LineaA,LineaA_Flores_Este_Turn02,Flores,1,0,0,1
1,2021,1/1/2021,08:00:00,08:15:00,LineaA,LineaA_SanPedrito_Oeste_Turn06,San Pedrito,0,0,2,2


In [8]:
#Auditoria de nulos
(
    df_2021.isna().sum().sort_values(ascending=False).head(10),
    df_2021.shape
)

(periodo            0
 fecha              0
 desde              0
 hasta              0
 linea              0
 molinete           0
 estacion           0
 pax_pagos          0
 pax_pases_pagos    0
 pax_franq          0
 dtype: int64,
 (8071680, 11))

### Importante

Se hizo el mismo procedimiento que con la data del 2020. Usar de esquema al 2019 para saber la estructura de columnas y su data y luego el reemplazo del nombre de las variables para coincidir con los demas archivos. 
Por suerte luego al correr el shape y la suma de nulos , no hay problemas con la data como si ocurrió con el 2020.

In [9]:
s = df_2021["fecha"].astype("string").str.strip()

dt_dmy = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
dt_iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")

df_2021["fecha_dt"] = dt_dmy.fillna(dt_iso)

print("NaT final:", df_2021["fecha_dt"].isna().sum())


NaT final: 0


In [10]:
df_2021["fecha"] = df_2021["fecha_dt"].dt.strftime("%Y-%m-%d")
df_2021 = df_2021.drop(columns=["fecha_dt"])


In [11]:
df_2021.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,2021,2021-01-01,08:00:00,08:15:00,LineaA,LineaA_Flores_Este_Turn02,Flores,1,0,0,1
1,2021,2021-01-01,08:00:00,08:15:00,LineaA,LineaA_SanPedrito_Oeste_Turn06,San Pedrito,0,0,2,2
2,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_Alem_N_Turn01,Leandro N. Alem,1,0,0,1
3,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_JMRosas_Oeste_Turn05,Rosas,1,0,0,1
4,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_Malabia_N_Turn02,Malabia,1,0,0,1


In [12]:
#Creacion de celdas faltantes
df_2021["desde"] = df_2021["desde"].astype("string").str.strip().replace("", pd.NA)
df_2021["hasta"] = df_2021["hasta"].astype("string").str.strip().replace("", pd.NA)

df_2021["hora_desde"] = pd.to_numeric(
    df_2021["desde"].str.split(":").str[0],
    errors="coerce"
).astype("Int8")

df_2021["hora_hasta"] = pd.to_numeric(
    df_2021["hasta"].str.split(":").str[0],
    errors="coerce"
).astype("Int8")
df_2021.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta
0,2021,2021-01-01,08:00:00,08:15:00,LineaA,LineaA_Flores_Este_Turn02,Flores,1,0,0,1,8,8
1,2021,2021-01-01,08:00:00,08:15:00,LineaA,LineaA_SanPedrito_Oeste_Turn06,San Pedrito,0,0,2,2,8,8
2,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_Alem_N_Turn01,Leandro N. Alem,1,0,0,1,8,8
3,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_JMRosas_Oeste_Turn05,Rosas,1,0,0,1,8,8
4,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_Malabia_N_Turn02,Malabia,1,0,0,1,8,8


In [13]:
s = df_2021["fecha"].astype("string").str.strip()

dt_dmy = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
dt_iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")

df_2021["fecha_dt"] = dt_dmy.fillna(dt_iso)
print("NaT final:", df_2021["fecha_dt"].isna().sum())


NaT final: 0


In [14]:
df_2021["dia_semana"] = df_2021["fecha_dt"].dt.day_name()
df_2021["mes"] = df_2021["fecha_dt"].dt.month.astype("Int8")
df_2021["dia_mes"] = df_2021["fecha_dt"].dt.day.astype("Int8")
df_2021["es_fin_semana"] = df_2021["fecha_dt"].dt.weekday.isin([5, 6]).astype("Int8")


In [15]:
df_2021["fecha"] = df_2021["fecha_dt"].dt.strftime("%Y-%m-%d")
df_2021 = df_2021.drop(columns=["fecha_dt"])
df_2021.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta,dia_semana,mes,dia_mes,es_fin_semana
0,2021,2021-01-01,08:00:00,08:15:00,LineaA,LineaA_Flores_Este_Turn02,Flores,1,0,0,1,8,8,Friday,1,1,0
1,2021,2021-01-01,08:00:00,08:15:00,LineaA,LineaA_SanPedrito_Oeste_Turn06,San Pedrito,0,0,2,2,8,8,Friday,1,1,0
2,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_Alem_N_Turn01,Leandro N. Alem,1,0,0,1,8,8,Friday,1,1,0
3,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_JMRosas_Oeste_Turn05,Rosas,1,0,0,1,8,8,Friday,1,1,0
4,2021,2021-01-01,08:00:00,08:15:00,LineaB,LineaB_Malabia_N_Turn02,Malabia,1,0,0,1,8,8,Friday,1,1,0


In [16]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8071680 entries, 0 to 8071679
Data columns (total 17 columns):
 #   Column           Dtype 
---  ------           ----- 
 0   periodo          int64 
 1   fecha            object
 2   desde            string
 3   hasta            string
 4   linea            object
 5   molinete         object
 6   estacion         object
 7   pax_pagos        int64 
 8   pax_pases_pagos  int64 
 9   pax_franq        int64 
 10  total            int64 
 11  hora_desde       Int8  
 12  hora_hasta       Int8  
 13  dia_semana       object
 14  mes              Int8  
 15  dia_mes          Int8  
 16  es_fin_semana    Int8  
dtypes: Int8(5), int64(5), object(5), string(2)
memory usage: 816.0+ MB


In [17]:
df_2021.shape

(8071680, 17)

In [18]:
#Ordenar por las dudas
df_2021 = df_2021[schema_2019]

In [19]:
df_2021.to_csv(
    path_clean + "historico_2021_clean.csv",
    index=False,
    sep=",",
    encoding="utf-8-sig"
)

print("ETL 2021 completado y exportado en la carpeta " + path_clean)

ETL 2021 completado y exportado en la carpeta ../data/clean/
